# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [7]:
class matrix:
    """
    checklist:
    two initialization modes
    indexing and slicing
    shape, transpose, row, column, block
    assignment
    scalar multiplication
    matrix addition/subtraction
    matrix multiplication
    element-wise multiplication
    equality comparison
    full operator overloading
    """

    # initializing
    
    def __init__(self, *args):

        # matrix(n, m) -> zero matrix
        
        if len(args) == 2:
            n, m = args

            # validate dimensions are actually numbers
            if not isinstance(n, int) or not isinstance(m, int):
                raise TypeError("Dimensions must be integers.")
            if n <= 0 or m <= 0:
                raise ValueError("Dimensions must be positive.")

            self.rows = n
            self.cols = m
            # create nested list filled with zeros
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        # matrix(list_of_lists)
        
        elif len(args) == 1:
            data = args[0]

            # check if input is a valid 2D list
            if not isinstance(data, list) or not all(isinstance(r, list) for r in data):
                raise TypeError("Input must be list of lists.")

            if len(data) == 0:
                raise ValueError("Matrix cannot be empty.")

            # check that all rows are the same length
            row_length = len(data[0])
            for r in data:
                if len(r) != row_length:
                    raise ValueError("All rows must have same number of columns.")

            self.rows = len(data)
            self.cols = row_length
            # store a deep copy of the data
            self.data = [r[:] for r in data]

        else:
            raise ValueError("Invalid initialization.")

    # basic methods
    
    def shape(self):
        # return dimensions as a tuple
        return (self.rows, self.cols)

    def to_list(self):
        # return the raw data as a list of lists
        return [r[:] for r in self.data]

    def transpose(self):
        # flip rows and columns
        return matrix([[self.data[i][j] for i in range(self.rows)]
                       for j in range(self.cols)])

    def row(self, n):
        # grab a specific row as a new matrix
        if n < 0 or n >= self.rows:
            raise IndexError("Row index out of range.")
        return matrix([self.data[n][:]])

    def column(self, n):
        # grab a specific column as a new matrix
        if n < 0 or n >= self.cols:
            raise IndexError("Column index out of range.")
        return matrix([[self.data[i][n]] for i in range(self.rows)])

    def block(self, n0, n1, m0, m1):
        # extract a sub-matrix from indices n0:n1 and m0:m1
        if not (0 <= n0 <= n1 <= self.rows):
            raise IndexError("Row indices out of range.")
        if not (0 <= m0 <= m1 <= self.cols):
            raise IndexError("Column indices out of range.")
        return matrix([self.data[i][m0:m1] for i in range(n0, n1)])

    # indexing and slicing
    
    def __getitem__(self, key):
        # handle matrix[i, j] or matrix[i:j, k:l]
        if isinstance(key, tuple):
            r, c = key

            # check if we are slicing rows or columns
            if isinstance(r, slice) or isinstance(c, slice):
                rows = range(*r.indices(self.rows)) if isinstance(r, slice) else [r]
                cols = range(*c.indices(self.cols)) if isinstance(c, slice) else [c]
                return matrix([[self.data[i][j] for j in cols] for i in rows])

            # standard integer indexing
            return self.data[r][c]

        # handle single row slicing matrix[i:j]
        elif isinstance(key, slice):
            rows = range(*key.indices(self.rows))
            return matrix([self.data[i][:] for i in rows])

        # standard single row access matrix[i]
        else:
            return self.data[key]

    def __setitem__(self, key, value):
        # update values using matrix[i, j] = x
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    # assignment
    
    def assign(self, other):
        # copy values from another matrix or list into this one
        if isinstance(other, matrix):
            if self.shape() != other.shape():
                raise ValueError("Matrix sizes must match.")
            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other.data[i][j]

        elif isinstance(other, list):
            if len(other) != self.rows:
                raise ValueError("Row count mismatch.")
            for r in other:
                if len(r) != self.cols:
                    raise ValueError("Column count mismatch.")
            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other[i][j]
        else:
            raise TypeError("Assignment must be matrix or list-of-lists.")

    # arithmetic methods
    
    def scalarmul(self, c):
        # multiply every element by a constant
        return matrix([[c * self.data[i][j] for j in range(self.cols)]
                       for i in range(self.rows)])

    def add(self, other):
        # basic matrix addition
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must match.")
        return matrix([[self.data[i][j] + other.data[i][j]
                        for j in range(self.cols)]
                       for i in range(self.rows)])

    def sub(self, other):
        # basic matrix subtraction
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must match.")
        return matrix([[self.data[i][j] - other.data[i][j]
                        for j in range(self.cols)]
                       for i in range(self.rows)])

    def mat_mult(self, other):
        # standard dot product multiplication
        if self.cols != other.rows:
            raise ValueError("Inner dimensions must agree.")

        result = matrix(self.rows, other.cols)

        # triple loop for row-by-column multiplication
        for i in range(self.rows):
            for j in range(other.cols):
                total = 0
                for k in range(self.cols):
                    total += self.data[i][k] * other.data[k][j]
                result[i, j] = total

        return result

    def element_mult(self, other):
        # multiply matching elements (Hadamard product)
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must match.")
        return matrix([[self.data[i][j] * other.data[i][j]
                        for j in range(self.cols)]
                       for i in range(self.rows)])

    def equals(self, other):
        # check if dimensions and all elements are identical
        if self.shape() != other.shape():
            return False
        for i in range(self.rows):
            for j in range(self.cols):
                if self.data[i][j] != other.data[i][j]:
                    return False
        return True

    # operator overloading
    
    def __add__(self, other):
        # overload +
        return self.add(other)

    def __sub__(self, other):
        # overload -
        return self.sub(other)

    def __mul__(self, other):
        # overload * (handles matrix * matrix and matrix * scalar)
        if isinstance(other, matrix):
            return self.mat_mult(other)
        elif isinstance(other, (int, float)):
            return self.scalarmul(other)
        else:
            raise TypeError("Unsupported multiplication.")

    def __rmul__(self, other):
        # overload * for scalar * matrix (reverse)
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        else:
            raise TypeError("Unsupported multiplication.")

    def __eq__(self, other):
        # overload ==
        return self.equals(other)

    # printing
    
    def __repr__(self):
        # make it look clean when printing
        return "\n".join(str(r) for r in self.data)

# special matrix constructors 

def constant(n, m, c):
    # matrix of size n x m filled with c
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    # matrix filled with 0.0
    return constant(n, m, 0.0)

def ones(n, m):
    # matrix filled with 1.0
    return constant(n, m, 1.0)

def eye(n):
    # identity matrix of size n
    I = zeros(n, n)
    for i in range(n):
        I[i, i] = 1.0
    return I

# demonstration of matrix properties

print("matrix property demonstration\n")

A = matrix([[1,2],
            [3,4]])

B = matrix([[0,1],
            [1,0]])

C = matrix([[2,0],
            [0,2]])

I = eye(2)

# checking standard linear algebra rules
print("(AB)C == A(BC):", (A*B)*C == A*(B*C)) # associativity
print("A(B+C) == AB+AC:", A*(B+C) == (A*B)+(A*C)) # distributivity
print("AB == BA:", A*B == B*A) # commutativity
print("AI == A:", A*I == A) # identity property

matrix property demonstration

(AB)C == A(BC): True
A(B+C) == AB+AC: True
AB == BA: False
AI == A: True
